In [ ]:
#| default_exp core

In [ ]:
#| export
import multiprocessing as mp
mp.set_start_method("spawn", force=True)

import os
import io
import re
import random
import base64
from io import BytesIO

import time
from datetime import timedelta

import numpy as np

import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from IPython.display import SVG

from PIL import Image as PILImage

import cv2
import pandas as pd
import json


from diffusers import StableDiffusionPipeline
from transformers import AutoProcessor, AutoModel

import vtracer



## Competition Metric Helpers

We also want to evaluate metrics of the original bitmap before converting to svg. Let’s implement it using [metric package](https://www.kaggle.com/code/jiazhuang/svg-image-fidelity).

In [ ]:
import numpy as np
import statistics
import pandas as pd

def image_resize(image, size=(384, 384)):
    return image.convert('RGB').resize(size)

def bitmap_score_instance_impl(multiple_choice_qa, image, random_seed=42):
    rng = np.random.RandomState(random_seed)
    group_seed = rng.randint(0, np.iinfo(np.int32).max)
    image_processor = metric.ImageProcessor(image=image_resize(image), seed=group_seed).apply()
    image = image_processor.image.copy()
    questions = multiple_choice_qa['question']
    choices = multiple_choice_qa['choices']
    answers = multiple_choice_qa['answer']
    aesthetic_score = metric.aesthetic_evaluator.score(image)
    vqa_score = metric.vqa_evaluator.score(questions, choices, answers, image)
    image_processor.reset().apply_random_crop_resize().apply_jpeg_compression(quality=90)
    ocr_score = metric.vqa_evaluator.ocr(image_processor.image)
    instance_score = metric.harmonic_mean(vqa_score, aesthetic_score, beta=0.5) * ocr_score
    return instance_score, vqa_score, ocr_score, aesthetic_score

def bitmap_score_instance(multiple_choice_qa, image, random_seed=42):
    is_single = not isinstance(image, list)
    if is_single:
        multiple_choice_qa = [multiple_choice_qa]
        image = [image]
    
    assert len(multiple_choice_qa) == len(image)

    results = []
    score_df = []
    for one_image, one_multiple_choice_qa in zip(image, multiple_choice_qa, strict=True):
        instance_score, vqa_score, ocr_score, aesthetic_score = bitmap_score_instance_impl(one_multiple_choice_qa, one_image, random_seed=42)
        results.append(instance_score)
        score_df.append([instance_score, vqa_score, ocr_score, aesthetic_score])

    fidelity = statistics.mean(results)
    score_df = pd.DataFrame(score_df, columns=['competition_score', 'vqa_score', 'ocr_score', 'aesthetic_score'])
    if is_single:
        return score_df.iloc[0].to_dict()
    else:
        return float(fidelity), score_df

## load model

In [ ]:
#| export
import torch
from diffusers import DiffusionPipeline, DPMSolverSinglestepScheduler, DDIMScheduler
from diffusers import FluxTransformer2DModel
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig, FluxTransformer2DModel, FluxPipeline, GGUFQuantizationConfig
from transformers import BitsAndBytesConfig as BitsAndBytesConfig, T5EncoderModel
from diffusers import AutoencoderKL, AutoencoderTiny
from diffusers.hooks import apply_group_offloading
import gc

# path for sdxl
flash_path    = 'sdxl-flash'
sdxl_path     = 'stable-diffusion-xl-base-1.0'
lora_path     = 'lora'
hypersd_path  = 'Hyper-SD'
# path for flux
t5_path = 't5-nf4'
flux_path  = "black-forest-lab/FLUX.1-schnell" 
vae_path  = 'taef1'
transformer = ''

def load_sdxl():
    # Load and attach DDIM scheduler to base pipeline
    scheduler = DDIMScheduler.from_pretrained(sdxl_path, subfolder="scheduler", timestep_spacing="trailing")
    
    base = DiffusionPipeline.from_pretrained(
        sdxl_path, torch_dtype=torch.float16, variant="fp16", scheduler=scheduler, use_safetensors=True
    )
    
    # load hyper sd
    base.load_lora_weights(hypersd_path, weight_name= "Hyper-SDXL-12steps-CFG-lora.safetensors")
    base.fuse_lora()
    
    # add lora ====================================================
    
    # # Load LoRA weights (local path or Hugging Face repo ID)
    base.load_lora_weights(lora_path, weight_name='Vector_illustration_XL.safetensors')
    # # To activate LoRA: specify `scale` (0.0 to 1.0, default 1.0)
    base.fuse_lora(lora_scale=0.8)

    return base

def load_flash():
    base = DiffusionPipeline.from_pretrained(flash_path, torch_dtype=torch.float16, use_safetensors=True)
    base.scheduler = DPMSolverSinglestepScheduler.from_config(base.scheduler.config, timestep_spacing="trailing")
    
    # add lora ====================================================
    # # Load LoRA weights (local path or Hugging Face repo ID)
    base.load_lora_weights(lora_path, weight_name='Vector_illustration_XL.safetensors')
    # # To activate LoRA: specify `scale` (0.0 to 1.0, default 1.0)
    base.fuse_lora(lora_scale=0.8)
    return base

def load_flux():
    # ============================= load trnsformer ====================================================
    # make sure you have the model_index.json in directory w/ this exact name: 'black-forest-labs/FLUX.1-schnell'
    # transformer = FluxTransformer2DModel.from_single_file(
    #     '/'.join([transformer,'flux1-schnell-Q2_K.gguf']),
    #     quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
    #     torch_dtype=torch.bfloat16,
    #     local_files_only=True,
    #     # config: A path to a directory containing the pipeline component configs in Diffusers format
    #     # this is important, otherwise it will look for it in default path or on huggingface(online)
    #     config = '/'.join([flux_path,'transformer']) 
    # )
    
    nf4_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    transformer = FluxTransformer2DModel.from_pretrained(
        flux_path,
        subfolder="transformer",
        quantization_config=nf4_config,
        torch_dtype=torch.bfloat16
    ).to('cpu')
    
    # Configure bnb 4bit for T5
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    # Load T5 with bnb 4bit
    t5_nf4 = T5EncoderModel.from_pretrained(t5_path,quantization_config=bnb_config,torch_dtype=torch.bfloat16).to('cpu')
    
    vae = AutoencoderTiny.from_pretrained(vae_path, torch_dtype=torch.bfloat16).to('cpu')
    
    base_flux = FluxPipeline.from_pretrained(
        pretrained_model_name_or_path=flux_path,
        transformer=transformer,
        vae = vae,
        text_encoder_2 = t5_nf4,
        torch_dtype=torch.bfloat16,
        local_files_only=True,
    ).to('cpu')

    # base.vae.enable_tiling()
    base_flux.enable_vae_slicing()
    gc.collect()
    torch.cuda.empty_cache()
    
    return base_flux

gen_configs = {
  'flash': dict(width=768, height=768, num_inference_steps=8, guidance_scale=3, num_images_per_prompt=1),
  'sdxl' : dict(width=768, height=768, num_inference_steps=12, guidance_scale=7.5, num_images_per_prompt=1),
  'flux' : dict(width=512, height=512, num_inference_steps=1, guidance_scale=0,   num_images_per_prompt=3),
}

    

## load MP

In [ ]:
%%writefile dllm.py
import torch
from multiprocessing import Process, Manager
from io import BytesIO
from PIL import Image as PILImage

# ——— 1) Placeholders for CPU-loaded pipelines & their names ———
_pipe0 = None
_pipe1 = None
_name0 = None
_name1 = None

# ——— 2) Shared negative prompt ———
negative_prompt = (
    'text, logo, mirror reflection, high-reflective, lines, deformed, ugly, '
    'wrong proportion, low res, bad anatomy, worst quality, low quality, '
    'framing, hatching, patterns, outlines'
)

# ——— 3) Offload toggle for all three models ———
offload_models = {}  # {'flash':bool,'sdxl':bool,'flux':bool}

# job → per‐rank attempts mapping
_job_attempts = {}

# ——— 4) Generation configs dict ———
_gen_configs = {}

def set_gen_configs(configs: dict):
    """
    configs: { model_name: {width:…, height:…, ...}, … }
    """
    global _gen_configs
    _gen_configs = configs

def set_pipelines(pipe0, name0, pipe1, name1):
    """
    Inject the two CPU-loaded pipelines and their model names.
    """
    global _pipe0, _pipe1, _name0, _name1
    _pipe0, _name0 = pipe0, name0
    _pipe1, _name1 = pipe1, name1

def set_offload_models(config):
    global offload_models
    offload_models = config

def worker_loop(rank, in_queue, out_queue,
                pipe0, name0, pipe1, name1,
                gen_configs, offload_models):
    torch.cuda.set_device(rank)
    pipe       = pipe0 if rank == 0 else pipe1
    model_name = name0 if rank == 0 else name1

    # move main pipeline to GPU
    pipe = pipe.to(rank)

    # always move Flux submodules to GPU
    if model_name == 'flux':
        pipe.transformer.to(rank)
        pipe.text_encoder_2.to(rank)
        if hasattr(pipe, 'vae'):
            pipe.vae.to(rank)

    # compile + optional CPU offload for flash/sdxl
    if model_name in ('flash','sdxl'):
        pipe.unet = torch.compile(pipe.unet, backend="eager", fullgraph=True)
        if offload_models.get(model_name, False):
            pipe.enable_model_cpu_offload()

    # flux: optional group offload
    elif model_name == 'flux' and offload_models.get('flux', False):
        from diffusers.hooks import apply_group_offloading
        apply_group_offloading(
            pipe.text_encoder_2,
            offload_device=torch.device('cpu'),
            onload_device=torch.device(f'cuda:{rank}'),
            offload_type='leaf_level',
            use_stream=True,
            record_stream=True
        )

    torch.cuda.empty_cache()

    # main loop: receive (prompt, tag, attempts)
    while True:
        item = in_queue.get()
        if item is None:
            break
        prompt, tag, attempts = item
        local_attempts = attempts.get(rank, 1)
        cfg = gen_configs[model_name]
        for _ in range(local_attempts):
            imgs = pipe(prompt, negative_prompt=negative_prompt, **cfg).images
            for img in imgs:
                buf = BytesIO()
                img.save(buf, format="PNG")
                out_queue.put((rank, tag, buf.getvalue()))

def start_workers(world_size=2):
    assert _pipe0 is not None and _pipe1 is not None, "Must call set_pipelines() first"
    assert _gen_configs, "Must call set_gen_configs() first"
    mgr       = Manager()
    in_queues = [mgr.Queue() for _ in range(world_size)]
    out_queue = mgr.Queue()
    procs     = []
    for r in range(world_size):
        p = Process(
            target=worker_loop,
            args=(
                r,
                in_queues[r],
                out_queue,
                _pipe0, _name0,
                _pipe1, _name1,
                _gen_configs,
                offload_models
            ),
            daemon=True
        )
        p.start()
        procs.append(p)
    return in_queues, out_queue, procs

def submit_to_all(prompt0, prompt1, in_queues, tag=None, attempts=None):
    import time
    if tag is None:
        tag = str(int(time.time()*1000))
    if attempts is None:
        attempts = {0:1,1:1}
    _job_attempts[tag] = attempts
    for rank, q in enumerate(in_queues):
        text = prompt0 if rank==0 else prompt1
        q.put((text, tag, attempts))
    return tag

def get_results(out_queue, world_size):
    r, tag, b = out_queue.get()
    attempts = _job_attempts[tag]
    cfg0 = _gen_configs[_name0]
    cfg1 = _gen_configs[_name1]
    total0 = attempts.get(0,1) * cfg0['num_images_per_prompt']
    total1 = attempts.get(1,1) * cfg1['num_images_per_prompt']

    imgs0, imgs1 = [], []
    img = PILImage.open(BytesIO(b))
    (imgs0 if r==0 else imgs1).append(img)
    while len(imgs0)<total0 or len(imgs1)<total1:
        rr, _, bb = out_queue.get()
        i = PILImage.open(BytesIO(bb))
        (imgs0 if rr==0 else imgs1).append(i)

    del _job_attempts[tag]
    return imgs0 + imgs1

def stop_workers(in_queues, procs):
    for q in in_queues:
        q.put(None)
    for p in procs:
        p.join()


In [ ]:
#| export
import dllm
from dllm import start_workers, submit_to_all, get_results, stop_workers
base_1, name1 = load_flash(), 'flash'
base_2, name2 = load_flash(), 'flash'
# base_2, name2 = load_sdxl(),  'sdxl'
# base_2, name2 = load_flux(),  'flux'

dllm.set_pipelines(base_1, name1, base_2, name2)
dllm.set_gen_configs(gen_configs)
dllm.set_offload_models({'flash':False,'sdxl':False,'flux':False})

# Look inside the dllm module, not your notebook globals:
print(dllm._pipe1) 


## load metric

In [ ]:
#| export
import metric

## load classifier

In [ ]:
# #| export
# from transformers import pipeline

# # Load once globally
# clf_path = kagglehub.model_download('jerry4083/bart-large-mnli/Transformers/default/1')
# zero_shot_classifier = pipeline("zero-shot-classification", model=clf_path, device=1)  # or device=-1 for CPU

# def classify_prompt(raw_prompt: str) -> str:
#     try:
#         candidate_labels = [
#             "landscape (scenery, nature, outdoor environments)",
#             "fashion (clothing, accessories, wearable items, anything human can wear)",
#             "geometry (abstract shapes, patterns, colors, layouts)"
#         ]
        
#         result = zero_shot_classifier(raw_prompt, candidate_labels)
#         top_label = result["labels"][0]

#         if "landscape" in top_label:
#             return "1"
#         elif "fashion" in top_label:
#             return "2"
#         elif "geometry" in top_label:
#             return "3"
#         else:
#             return "0"
#     except Exception as e:
#         print(f"[Prompt fallback] Failed to classify prompt: {e}")
#         return "0"

## load llm

In [ ]:
# #| export
# from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, GenerationConfig
# import torch

# # model_id = kagglehub.model_download('jerry4083/qwen3-1.7b-unsloth-bnb-4bit/Transformers/default/1')
# model_id = kagglehub.model_download('qwen-lm/qwen-3/Transformers/0.6b/1')

# quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

# llm_model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     quantization_config=quantization_config
# ).eval().to("cuda:1")

# llm_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# gen_config = GenerationConfig(
#     do_sample=True,
#     temperature=0.8,
#     top_p=0.95,
#     max_new_tokens=256,
#     pad_token_id=llm_tokenizer.eos_token_id,
#     eos_token_id=llm_tokenizer.eos_token_id,
# )

# torch.cuda.empty_cache()

## classify prompt

In [ ]:
#| export
def classify_prompt(raw_prompt: str) -> str:
    messages = [
        {
            "role": "system",
            "content": """You are a classification assistant. Given a short prompt and based on the main subjects it describes, classify it into one of the following most related categories:

1 = landscape (scenery, nature, outdoor environments)
2 = fashion (clothing, accessories, wearable items, anything human can wear)
3 = geometry (abstract shapes, patterns, colors, layouts)

Your return should be one number(1, 2, or 3) only. Return nothing else.
"""
        },
        {
            "role": "user",
            "content": """Example: 
prompt:'a purple forest at dusk' -> category: 1
prompt: 'gray wool coat with a faux fur collar' -> category: 2
prompt: 'khaki triangles and azure crescents' -> category: 3

Prompt: 'a lighthouse overlooking the ocean'. category:
"""
        },
        {
            "role": "assistant",
            "content": "1"
        },
        {
            "role": "user",
            "content": f"prompt: {raw_prompt}. category:"
        },
    ]

    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            enable_thinking=False,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device)
        # print(inputs)
        # print(type(inputs))
    
        with torch.inference_mode():
            outputs = llm_model.generate(**inputs, generation_config=gen_config)
    
        outputs = outputs[:, inputs.input_ids.shape[1]:]
        outputs = llm_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
        # print(outputs)
        # print(type(outputs))
    
        # outputs[0] is the full chat + elaboration
        return outputs[0].strip()
    
    except Exception as e:
        print(f"[Prompt fallback] Failed to elaborate prompt: {e}")
        output = '0'
        return output

## elaborate prompt_landscape

In [ ]:
#| export
def elaborate_prompt_scenery(raw_prompt: str) -> str:
    default_output = f'Create a stylized geometric illustration of {raw_prompt} with geometric form and abstract silhouettes. The composition uses smooth color blocks, subtle line art, and a dreamlike, flattened perspective to emphasize elegance and simplicity. Inspired by minimalist vector art, with vibrant color and cinematic lighting. The overall atmosphere is tranquil yet powerful, with strong color contrasts, 8K, masterpiece, trending at artstation.'
    if len(raw_prompt.strip().split()) > 10:
        print("Prompt is too long.")
        return default_output

    
    messages = [
        {
            "role": "system",
            "content": """You are a skilled prompt engineer for image generation. You need to convert a short raw prompt into a stylized descriptive sentence suitable for generating sticker in Flux.1.
The prompt format is: A stylized ...., rendered in.... Simplified geometric....silhouettes....
Some rules you must follow when crafting the prompt:
1. Clearly define the subject and its attributes, and relationship between different elements. No new elements should be added.
2. Only return the elaborated prompt with nothing else.
"""
        },
        {
            "role": "user",
            "content": """Elaborate a prompt for me so that I can use it for generating vector-art image. 
Some examples:
'a purple forest at dusk' -> 'A stylized abstract purple forest at dusk, rendered in soft pastel hues of lavender, indigo, and pale gold. Simplified geometric tree silhouettes with flowing, organic shapes, glowing gradient light effects in the sky.'
'gray wool coat with a faux fur collar' -> 'A stylized abstract gray wool coat with a faux fur collar on a lady, rendered in soft muted tones of charcoal, dove gray, and ivory. Simplified geometric tailoring defines the coat’s clean silhouette, while the collar is illustrated with flowing, feathery textures in layered light beige and cream, evoking warmth and refined elegance.'
'khaki triangles and azure crescents' -> 'A stylized abstract arrangement of khaki triangles and azure crescents, rendered in soft pastel hues of sand, olive, and sky blue. Simplified geometric forms are carefully balanced in a rhythmic composition, with subtle gradients and overlapping shapes creating a sense of motion and visual harmony against a light, neutral backdrop.'

Raw prompt: 'a lighthouse overlooking the ocean'. Your elaborated prompt:
"""
        },
        {
            "role": "assistant",
            "content": "A stylized lighthouse perched on a rocky cliff above a calm ocean, rendered in soft tones of misty blue, slate gray, and warm ivory. Simplified geometric forms and gentle gradients evoke a tranquil seascape at dawn, with minimal waves and a faint beam of light sweeping across the horizon."
        },
        {
            "role": "user",
            "content": f"raw prompt: {raw_prompt}. Your elaborated prompt:"
        },
    ]

    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            enable_thinking=False,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device)
        # print(inputs)
        # print(type(inputs))
    
        with torch.inference_mode():
            outputs = llm_model.generate(**inputs, generation_config=gen_config)
    
        outputs = outputs[:, inputs.input_ids.shape[1]:]
        outputs = llm_tokenizer.batch_decode(outputs, skip_special_tokens=True)
        output = outputs[0].strip()
        output = output + 'The composition uses smooth color blocks, subtle line art, and a dreamlike, flattened perspective to emphasize elegance and simplicity. Inspired by minimalist vector art, with vibrant color and cinematic lighting. The overall atmosphere is tranquil yet powerful, with strong color contrasts, 8K, masterpiece, trending at artstation.'
    
        # print(outputs)
        # print(type(outputs))
    
        # outputs[0] is the full chat + elaboration
        return output
    
    except Exception as e:
        print(f"[Prompt fallback] Failed to elaborate prompt: {e}")
        return default_output

## elaborate prompt_geometry

In [ ]:
#| export
def elaborate_prompt_geo(raw_prompt: str) -> str:
    default_output = f'Create a sticker of {raw_prompt} shown in a flat 2D front view against a pure white background.There is no shadows, reflection or depth, evoking a vectorized, minimalist design style.'
    if len(raw_prompt.strip().split()) > 10:
        print("Prompt is too long.")
        return default_output
        
    messages = [
        {
            "role": "system",
            "content": """You are a skilled prompt engineer for image generation. You need to convert a short raw prompt into a stylized descriptive sentence suitable for generating sticker in Flux.1."
The prompt format is: A sticker depicting...., shown in a flat 2D front view against a pure white background.... with no shadows, reflection or depth, evoking a vectorized, minimalist design style."
Some rules you must follow when crafting the prompt:
1. Clearly and precisely define the shape, color and relationship between each shape(defalut is no overlaping between different shapes)
2. Define the amount of each shapes. If it is single, call it one; if it is plural, give it a reasonable amount(default amount is three)
3. Pure white background, but if there is any shape in white color already, pick another neutral dark color as background.
4. Shape should be bold and clear and visible. For example, a thread should be a thick thread to be visible and not covered by other shape.
Only return the elaborated prompt with nothing else.
"""
        },
        {
            "role": "user",
            "content": """Elaborate a prompt for me so that I can use it for generating sticker. 
Some examples:
'khaki triangles and azure crescents' -> 'A sticker depicting three khaki triangles and two azure crescents, shown in a flat 2D front view against a pure white background. The triangles have bold, solid khaki fills with sharp corners, arranged in a loose cluster with slight angular variation, while the crescents are smooth, thick arcs in bright azure, placed nearby in curved contrast, with no shadows, reflection or depth, evoking a vectorized, minimalist design style.'
'purple pyramids spiraling around a bronze cone' -> 'A sticker depicting three purple pyramids spiraling around a bronze cone, shown in a flat 2D front view against a pure white background. The cone stands central with a solid flat bronze fill and clean edges, while three bold purple pyramids orbit in a loose spiral pattern, evenly spaced and clearly defined, with no shadows, reflection or depth, evoking a vectorized, minimalist design style.'
'crimson rectangles forming a chaotic grid' -> 'A sticker depicting seven crimson rectangles forming a chaotic grid, shown in a flat 2D front view against a pure white background. The rectangles have bold, solid crimson fills with clean black outlines, scattered and overlapping at irregular angles, creating a fragmented, unbalanced pattern, with no shadows, reflection or depth, evoking a vectorized, minimalist design style.'

Raw prompt: 'magenta trapezoids layered on a transluscent silver sheet'. Your elaborated prompt:
"""
        },
        {
            "role": "assistant",
            "content": "A sticker depicting three magenta trapezoids layered on one translucent grey color sheet, shown in a flat 2D front view against a pure white background. The trapezoids have bold, solid fills with sharp edges, stacked diagonally with slight overlap, while the grey color sheet appears as a soft, semi-transparent base, with no shadows, reflection or depth, evoking a vectorized, minimalist design style."
        },
        {
            "role": "user",
            "content": f"raw prompt: {raw_prompt}. Your elaborated prompt:"
        },
    ]

    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            enable_thinking=False,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(llm_model.device)
        # print(inputs)
        # print(type(inputs))
    
        with torch.inference_mode():
            outputs = llm_model.generate(**inputs, generation_config=gen_config)
    
        outputs = outputs[:, inputs.input_ids.shape[1]:]
        outputs = llm_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
        # print(outputs)
        # print(type(outputs))
    
        # outputs[0] is the full chat + elaboration
        return outputs[0].strip()
    
    except Exception as e:
        print(f"[Prompt fallback] Failed to elaborate prompt: {e}")
        return default_output

In [ ]:
%%time
prompt = 'a purple silk scarf with tassel trim'
prompt = elaborate_prompt_geo(prompt)
print(prompt)

## Load Data

In [ ]:
import pandas as pd
import json
train_df = pd.read_csv('train.csv')
train_question_df = pd.read_parquet('questions.parquet')

train_question_df = train_question_df.groupby('id').apply(lambda df: df.to_dict(orient='list'))
train_question_df = train_question_df.reset_index(name='qa')

train_question_df['question'] = train_question_df.qa.apply(lambda qa: json.dumps(qa['question'], ensure_ascii=False))

train_question_df['choices'] = train_question_df.qa.apply(
    lambda qa: json.dumps(
        [x.tolist() for x in qa['choices']], ensure_ascii=False
    )
)

train_question_df['answer'] = train_question_df.qa.apply(lambda qa: json.dumps(qa['answer'], ensure_ascii=False))

train_df = pd.merge(train_df, train_question_df, how='left', on='id')

train_df['multiple_choice_qa'] = train_df.apply(
    lambda r: {
    'question': json.loads(r.question),
    'choices': json.loads(r.choices),
    'answer': json.loads(r.answer)
    },
    axis=1,
)

# train_df.head()

# Image -> SVG

* Did a bunch of work here trying to get good results..

In [ ]:
#| export
import re
import vtracer
from IPython.display import SVG, display, Image
from PIL import Image as PILImage
input_path = "output.png"
output_path = "output.svg"

In [ ]:
#| export
default_svg_code = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""

In [ ]:
#| export
import re
def convert_paths_to_polygons(svg_code: str, size: int) -> str:
    # Find all path elements
    scale_factor = 384 / size

    path_pattern = re.compile(
        r'<path[^>]*d="([^"]+)"[^>]*fill="([^"]+)"[^>]*transform="translate\(([^)]+)\)"[^>]*/?>'
    )
    paths = path_pattern.findall(svg_code)

    polygons = []

    for d_content, fill_color, translate in paths:
        tx, ty = map(float, translate.split(','))
        points = []
        tokens = re.findall(r'[MLZmlz]|-?\d+\.?\d*,\-?\d+\.?\d*', d_content.strip())
        for token in tokens:
            if token in {'M', 'L', 'Z', 'm', 'l', 'z'}:
                continue
            x_str, y_str = token.split(',')
            x = int(round(float(x_str) + tx))
            y = int(round(float(y_str) + ty))
            points.append(f"{x},{y}")

        if points:
            polygon = f'<polygon points="{" ".join(points)}" fill="{fill_color}"/>'
            polygons.append(polygon)

    # Build the compact SVG
    new_svg = (
        f'<svg width="384" height="384" viewBox="0 0 384 384"><g transform="scale({scale_factor})">'
        + "".join(polygons)
        + '</g></svg>'
    )

    return new_svg


def fix_svg_size(svg_code: str, target_size: int = 384) -> str:
    """
    Update the <svg> tag's width and height to match target size.
    """
    # Replace width attribute
    svg_code = re.sub(
        r'(width\s*=\s*")([^"]+)(")',
        lambda m: f'{m.group(1)}{target_size}{m.group(3)}',
        svg_code,
        count=1
    )
    # Replace height attribute
    svg_code = re.sub(
        r'(height\s*=\s*")([^"]+)(")',
        lambda m: f'{m.group(1)}{target_size}{m.group(3)}',
        svg_code,
        count=1
    )
    return svg_code


def bitmap_to_svg_layered(img, input_path, output_path, resolution):
    default_svg = default_svg_code
    
    # Define input and output paths
    # (input_path and output_path are already passed as arguments)

    # Step 1: Resize the input image to 256x256
    # 256x256 is helpful bc each length od path is shorter, so with the same max_svg_length, we can have more paths(more color)
    size = resolution
    img = img.resize((size,size), PILImage.LANCZOS)
    img.save(input_path)

    # Step 2: Convert the resized image to SVG
    max_svg_length = 9996  # target length limit

    # Define injection to prevent OCR hallucination
    injection_a1 = '\n<path d="M5 374 L10 364 L15 374 M7 368 L13 368" stroke="white"/>'      # Bottom-left A
    injection_a2 = '\n<path d="M364 30 L370 20 L376 30 M367 25 L373 25" stroke="white"/>'         # Top-right A
    injection = injection_a1 + injection_a2
    injection_length = len(injection)

    # Binary search for best layer_difference
    low = 1
    high = 200
    best_svg_code = None
    best_layer_difference = 10

    try:
        while low <= high:
            layer_difference = (low + high) // 2
    
            vtracer.convert_image_to_svg_py(
                input_path,
                output_path,
                colormode='color',        # Options: 'color' or 'binary'
                hierarchical='stacked',   # Options: 'stacked' or 'cutout'
                mode='polygon',           # Options: 'spline', 'polygon', or 'none'
                filter_speckle=3,          # remove tiny regions
                color_precision=8,         # reduce color complexity
                layer_difference=layer_difference,  # more aggressive merging
                corner_threshold=10,       # remove subtle corners
                length_threshold=10,       # remove short paths
                max_iterations=10,         # faster, less detail
                splice_threshold=10,       # simplify curves
                path_precision=3           # reduce vertex detail
            )
    
            # Step 3: Read and display the SVG
            try:
                with open(output_path, "r", encoding="utf-8") as f:
                    svg_code = f.read()
            except UnicodeDecodeError:
                # Bad SVG output (probably corrupted), try again
                high = layer_difference - 1
                continue  # go back to binary search
    
            # Clean the first two lines if present
            lines = svg_code.splitlines()
            removed_length = 0
            if lines and lines[0].strip().startswith('<?xml'):
                removed_length += len(lines[0]) + 1  # +1 for newline
                lines = lines[1:]
            if lines and lines[0].strip().startswith('<!--'):
                removed_length += len(lines[0]) + 1  # +1 for newline
                lines = lines[1:]
            svg_code = "\n".join(lines)
    
            svg_code = svg_code.replace(
                '<svg ',
                '<svg viewBox="0 0 384 384" ', #this is 20 bytes more
                1  # only replace first occurrence
            )
            
            # remove version and xmlns attributes
            svg_code = re.sub(r'\s*version="[^"]*"', '', svg_code)
            svg_code = re.sub(r'\s*xmlns="[^"]*"', '', svg_code)
    
            # use polygon instead of path
            svg_code = convert_paths_to_polygons(svg_code, size)
                
            # Correct length check: give credit for removed lines
            if len(svg_code) + injection_length <= max_svg_length:
                best_svg_code = svg_code
                best_layer_difference = layer_difference
                high = layer_difference - 1  # search for even more detail
            else:
                low = layer_difference + 1  # simplify more


        # ============= if layer_diff = 200 is not enough, keep increasing ==========================================
        if best_svg_code == None:
            layer_difference = 210
            max_diff = 1000 
            while layer_difference <= max_diff:
                vtracer.convert_image_to_svg_py(
                    input_path,
                    output_path,
                    colormode='color',        # Options: 'color' or 'binary'
                    hierarchical='stacked',   # Options: 'stacked' or 'cutout'
                    mode='polygon',           # Options: 'spline', 'polygon', or 'none'
                    filter_speckle=3,          # remove tiny regions
                    color_precision=8,         # reduce color complexity
                    layer_difference=layer_difference,  # more aggressive merging
                    corner_threshold=10,       # remove subtle corners
                    length_threshold=10,       # remove short paths
                    max_iterations=10,         # faster, less detail
                    splice_threshold=10,       # simplify curves
                    path_precision=3           # reduce vertex detail
                )
        
                try:
                    with open(output_path, "r", encoding="utf-8") as f:
                        svg_code = f.read()
                except UnicodeDecodeError:
                    # Bad SVG output (probably corrupted), try again
                    continue  # go back to start of while loop
        
                # Clean the first two lines if present
                lines = svg_code.splitlines()
                removed_length = 0
                if lines and lines[0].strip().startswith('<?xml'):
                    removed_length += len(lines[0]) + 1  # +1 for newline
                    lines = lines[1:]
                if lines and lines[0].strip().startswith('<!--'):
                    removed_length += len(lines[0]) + 1  # +1 for newline
                    lines = lines[1:]
                svg_code = "\n".join(lines)
        
                svg_code = svg_code.replace(
                    '<svg ',
                    '<svg viewBox="0 0 384 384" ', #this is 20 bytes more
                    1  # only replace first occurrence
                )
                
                # remove version and xmlns attributes
                svg_code = re.sub(r'\s*version="[^"]*"', '', svg_code)
                svg_code = re.sub(r'\s*xmlns="[^"]*"', '', svg_code)
        
                # use polygon instead of path
                svg_code = convert_paths_to_polygons(svg_code, size)
                print(f'{len(svg_code)=}')
                    
                # Correct length check: give credit for removed lines
                if len(svg_code) + injection_length <= max_svg_length: # acount for injection
                    best_svg_code = svg_code
                    best_layer_difference = layer_difference
                    break
                else:
                    layer_difference += 10
    
        svg_code = fix_svg_size(best_svg_code, target_size=384) # doesnt change length
        # Inject fake letter path to prevent OCR hallucination
        svg_code = svg_code.replace("</svg>", injection + "</svg>")
        print(f'{best_layer_difference=}')

    except Exception as e:
        print(f"{e}")
        svg_code = default_svg

    
    return svg_code

In [ ]:
#| export
import os
from PIL import Image
import numpy as np
from skimage.color import rgb2lab
from scipy.stats import entropy
import matplotlib.pyplot as plt
def compute_color_richness_entropy(image, bins=32):
    # Ensure image is RGB and resized
    image = image.convert('RGB')
    image = image.resize((256, 256))
    img_array = np.array(image) / 255.0
    lab_image = rgb2lab(img_array)

    # Flatten LAB channels
    L = lab_image[:, :, 0].flatten()
    A = lab_image[:, :, 1].flatten()
    B = lab_image[:, :, 2].flatten()

    # Compute histograms
    L_hist, _ = np.histogram(L, bins=bins, density=True)
    A_hist, _ = np.histogram(A, bins=bins, density=True)
    B_hist, _ = np.histogram(B, bins=bins, density=True)

    # Calculate entropy for each channel
    L_entropy = entropy(L_hist + 1e-8)
    A_entropy = entropy(A_hist + 1e-8)
    B_entropy = entropy(B_hist + 1e-8)

    total_entropy = L_entropy + A_entropy + B_entropy

    # Normalize: range from 0-10.39
    max_entropy = 10.39 # = 3 * np.log(bins)
    normalized_entropy = total_entropy / max_entropy
    
    return normalized_entropy

def map_score_to_range(normalized_score):
    if normalized_score <= 0.5:
        return 384
    elif normalized_score >= 0.7:
        return 128
    else:
        # Linear interpolation between 384 and 128
        alpha = (normalized_score - 0.5) / (0.7 - 0.5)
        return int(384 - alpha * (384 - 128))

## SD evaluator

In [ ]:
#| export
from PIL import Image
import ast
import random
import spacy
import pandas as pd

# Load large spaCy model
nlp = spacy.load("en_core_web_lg")

# Extract grouped elements: [descriptive noun phrase] vs [spatial phrase]
def generate_qa_spacy(text):

    try:
        doc = nlp(text)
        descriptive_elements = set(chunk.text for chunk in doc.noun_chunks)
    
        spatial_phrases = set()
        for token in doc:
            if token.dep_ == "prep":
                phrase = token.text
                # Get all children of the preposition (usually includes noun + modifiers)
                object_phrase = " ".join([child.text for child in token.children])
                if object_phrase:
                    phrase = f"{phrase} {object_phrase}"
                spatial_phrases.add(phrase)
    
        questions = []
        for i in list(descriptive_elements):
            question = 'Are there ' + i + ' in the image?'
            questions.append(question)
        for i in list(spatial_phrases):
            question = 'Are there something ' + i + '?' 
            questions.append(question)

        if not questions:
            return generate_qa(text)

        if len(questions) == 1:
            q = f"Does this image look like: '{text}'?"
            questions.append(q)
    
        if len(questions) > 3:
                questions = random.sample(questions, 3)
    
        # Build the QA dictionary
        qa = {
            "question": questions,
            "choices": [["no", "yes"]] * len(questions),
            "answer": ["yes"] * len(questions)
        }
        return qa
        
    except Exception as e:
        print(f"[Prompt fallback] Failed to create qa: {e}")
        return generate_qa(text)


def generate_qa(prompt: str) -> dict:
    """
    Generate VQA-style question-answer dict based on a given prompt.
    Includes yes/no questions and one clarity rating.
    """
    qa = {
        'question': [
            f"Are there {prompt} in the image?",
            f"Does this image look like: {prompt}?"
        ],
        'choices': [
            ['no', 'yes'],
            ['no', 'yes']
        ],
        'answer': [  
            'yes',   
            'yes' 
        ]
    }
    return qa
    
def image_resize(image, size=(384, 384)):
    return image.convert('RGB').resize(size)

def get_score(sample, qa, vqa = True, ocr=False):

    try:
        # If sample is a string, treat as SVG and convert to image
        rng = np.random.RandomState(42)
        group_seed = rng.randint(0, np.iinfo(np.int32).max)
        if isinstance(sample, str):
            image = metric.svg_to_png(sample)
        else:
            image = sample
        
        image_processor = metric.ImageProcessor(image=image_resize(image), seed=group_seed).apply()
        image = image_processor.image.copy()
    
        try:
            aesthetic_score = metric.aesthetic_evaluator.score(image)
        except Exception as e:
            print(f"AES score error: {e}")
            aesthetic_score = 0.5

        if vqa:
            try:
                questions = qa['question']
                choices = qa['choices']
                answers = qa['answer']
                vqa_score = metric.vqa_evaluator.score(questions, choices, answers, image)
            except Exception as e:
                # raise
                print(f"VQA score error: {e}")
                vqa_score = 0.5
        else:
            vqa_score = 0.5
        
        # if ocr:
        #     image_processor.reset().apply_random_crop_resize().apply_jpeg_compression(quality=90)
        #     ocr_score = metric.vqa_evaluator.ocr(image_processor.image)
        # else:
        #     ocr_score = 1.0
        ocr_score = 1.0
    
        instance_score = metric.harmonic_mean(vqa_score, aesthetic_score, beta=0.5) * ocr_score
    
        return instance_score, aesthetic_score, ocr_score, vqa_score
    
    except Exception as e:
        # raise
        print(f"score error: {e}")
        return 0.5, 0.5, 1.0, 0.5

 




In [ ]:
print(generate_qa_spacy('crimson rectangles forming a chaotic grid'))

## Implement the package Model class

In [ ]:
#| export
default_svg_code = """<svg width="384" height="384" viewBox="0 0 384 384"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""

In [ ]:
#| export
import time

class Model:

    def __init__(self):
        # Fallback if prompt is empty
        self.default_svg = (
            '<svg width="384" height="384" viewBox="0 0 384 384">'
            '<circle cx="50" cy="50" r="40" fill="red" />'
            '</svg>'
        )

        # Prompt formatting
        self.prompt_prefix = "a stylized digital painting presenting a"
        self.prompt_suffix = (
            " from distance. The painting promote vector-art aesthetic, "
            "in watercolor art style with vibrant and clean background. "
            "The overall atmosphere is tranquil yet powerful, raw-photo "
            "hyper-detail, 4K, cinematic lighting, award-winning, masterpiece."
        )

        self._workers_started = False
        self.attempt_1 = 5
        self.attempt_2 = 5

        self.num_attempt = 1

    def _ensure_workers(self):
        if not self._workers_started:
            self.in_queues, self.out_queue, self.procs = start_workers(world_size=2)
            self._workers_started = True

    def gen_bitmaps(self, prompt_0: str, prompt_1:str):

        
        attempts = {0: self.attempt_1, 1: self.attempt_2}
        tag = submit_to_all(prompt_0, prompt_1, self.in_queues, attempts=attempts)
        bitmaps = get_results(self.out_queue, world_size=2)
        
        return bitmaps

    def predict_impl(self, prompt: str):
        
        if not prompt:
            return self.default_svg, None
            
        # ============================ start the workers here ===============================
        self._ensure_workers()
    
        # ============================ generate qa and prompts ===============================
        prompt_0 = "a stylized digital painting presenting a " + prompt + (
            " from distance. The painting promote vector-art aesthetic, "
            "in watercolor art style with vibrant and clean background. "
            "The overall atmosphere is tranquil yet powerful, raw-photo "
            "hyper-detail, 4K, cinematic lighting, award-winning, masterpiece."
        )

        prompt_1 = "a simplified color icon presenting " + prompt + (
            " from distance, with clean silhouette and abstract geometric form. Vector-art sytle, cinematic lighting, award-winning"
        )
        qa = generate_qa_spacy(prompt)
        print(qa)

        # ============================ generate images ===============================
        bitmaps = []
        best_score = 0.0
        best_svg = None
        best_img = None
        start_ts = time.time()
        print(f"======  {prompt_0}  =======")
        print(f"======  {prompt_1}  =======")
        print(f"======  Generating.....  =======")
        for attempt in range(self.num_attempt):
            if time.time() - start_ts > 52:
                print(f"Timeout after {attempt} attempts.")
                break

            bmps = self.gen_bitmaps(prompt_0, prompt_1)
            bitmaps.extend(bmps)
        torch.cuda.empty_cache()

        # ============================ score images ===============================
        print(f"======  Scoring.....  =======")
        print(f'{len(bitmaps)=}')
        if len(bitmaps) != self.attempt_1 + self.attempt_2:
            raise RuntimeError(f"Generation failed")
            
        for bitmap in bitmaps:
            display(bitmap.resize((128,128)))
            
            # resolution from color score
            color_score = compute_color_richness_entropy(bitmap)
            resolution = map_score_to_range(color_score)
            print(f"color_score={color_score:.3f}, resolution={resolution}")

            # bitmap → SVG
            svg = bitmap_to_svg_layered(bitmap, input_path, output_path, resolution)

            # VQA/aesthetic scoring
            instance_score, aesthetic_score, ocr_score, vqa_score = get_score(sample=svg, qa=qa, ocr=False)
            score = instance_score
            print(f'{aesthetic_score =}, {vqa_score=}, {score=}')
            print('svg length:', len(svg))

            # keep best
            if score >= best_score:
                best_score, best_svg, best_img = score, svg, bitmap
                    
        print(f"Final best score: {best_score:.3f}")
        return best_svg, best_img

    def predict(self, prompt: str):
        svg, img = self.predict_impl(prompt)
        torch.cuda.empty_cache()
        return svg

    def close(self):
        stop_workers(self.in_queues, self.procs)


In [ ]:
#| export
model = Model()

In [ ]:
%%time
import time

r = train_df.iloc[10]
description = r.description
start = time.time()

# print(description)
svg, img = model.predict_impl(description)

end = time.time()
print(f"Time taken: {end - start:.2f} seconds")

display(img)
display(SVG(svg))
print(svg)

In [ ]:
display(img)
display(SVG(svg))
print(svg)

## test metric

In [ ]:


# for _ in range(10):
#     # qa = {'question': [], 'choices': [], 'answer': []}
#     # print(get_score(sample=svg, qa=qa, ocr=False))
#     qa = {'question': ['Is A smiling sloth wearing a leather jacket, a cowboy hat, a kilt and a bowtie. The sloth is holding a quarterstaff and a big book. Is A smiling sloth wearing a leather jacket, a cowboy hat, a kilt and a bowtie. The sloth is holding a quarterstaff and a big book.Is A smiling sloth wearing a leather jacket, a cowboy hat, a kilt and a bowtie. The sloth is holding a quarterstaff and a big book. The sloth stands a few feet in front of a shiny VW van. The van has a cityscape painted on it and parked on grass.?', 'Does the sloth have a cowboy hat?', 'Is the sloth holding a big book?', 'What is painted on the VW van?'], 'choices': [['a cityscape', 'flowers', 'mountains', 'the ocean'], ['no', 'yes'], ['no', 'yes'], ['a cityscape', 'flowers', 'mountains', 'the ocean']], 'answer': ['yes', 'yes', 'yes', 'a cityscape']}
#     print(get_score(sample=svg, qa=qa, ocr=False))
#     qa = {'question': ['Is A smiling sloth wearing a leather jacket?', 'Does the sloth have a cowboy hat?'], 'choices': [['a cityscape', 'flowers', 'mountains', 'the ocean'], ['no', 'yes']], 'answer': ['yes', 'yes',]}
#     print(get_score(sample=svg, qa=qa, ocr=False))

# metric.score_instance(r.multiple_choice_qa, svg, random_seed=42)


## Evaluate on train dataset (LB prediction!)

In [ ]:
%%time

import matplotlib.pyplot as plt
%matplotlib inline

import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

start = time.time()

train_df['raw_res'] = train_df.description.progress_apply(model.predict_impl)

end = time.time()
print(f"Time taken: {end - start:.2f} seconds")

train_df['svg'] = train_df.raw_res.apply(lambda x: x[0])
train_df['bitmap'] = train_df.raw_res.apply(lambda x: x[1])
train_df['bitmap_score'] = train_df.progress_apply(
    lambda r: bitmap_score_instance(r.multiple_choice_qa, r.bitmap, random_seed=42),
    axis=1,
)

train_df['svg_score'] = train_df.progress_apply(
    lambda r: metric.score_instance(r.multiple_choice_qa, r.svg, random_seed=42),
    axis=1,
)


print(f"Time taken: {end - start:.2f} seconds")

for r in train_df.itertuples():
    b_score = r.bitmap_score['competition_score']
    b_vqa = r.bitmap_score['vqa_score']
    b_ocr = r.bitmap_score['ocr_score']
    b_aesthetic = r.bitmap_score['aesthetic_score']

    s_score = r.svg_score['competition_score']
    s_vqa = r.svg_score['vqa_score']
    s_ocr = r.svg_score['ocr_score']
    s_aesthetic = r.svg_score['aesthetic_score']
    
    plt.figure(figsize=(12, 6))
    plt.suptitle(r.description, y=0.93)
    
    plt.subplot(1, 2, 1)
    plt.imshow(np.array(r.bitmap))
    plt.axis('off')
    plt.title(f'bitmap: score={b_score:.2f}, vqa={b_vqa:.2f}, ocr={b_ocr:.2f}, aes={b_aesthetic:.2f}')

    plt.subplot(1, 2, 2)
    plt.imshow(metric.svg_to_png(r.svg))
    plt.axis('off')
    plt.title(f'svg: score={s_score:.2f}, vqa={s_vqa:.2f}, ocr={s_ocr:.2f}, aes={s_aesthetic:.2f}')


mean_bitmap_score = pd.DataFrame(train_df['bitmap_score'].tolist()).mean(axis=0)
print(mean_bitmap_score)


mean_svg_score = pd.DataFrame(train_df['svg_score'].tolist()).mean(axis=0)
print(mean_svg_score)


print(f'Original bitmap score: {mean_bitmap_score.competition_score}')
print(f'Final svg score: {mean_svg_score.competition_score}')